# Timoshenko Causal SPINN Forward Problem (Demo)

Timoshenko beam, 11th vibration mode, fixed boundary, $\alpha = \beta = k = 1$.
Coupled 2nd-order system in displacement $u(t,x)$ and rotation $\theta(t,x)$:

$$
\theta_{tt} - \theta_{xx} - (u_x - \theta) = 0
$$
$$
u_{tt} - (u_{xx} - \theta_x) + u = \cos(t)
$$

Domain: $x \in [0, 11\pi]$, $t \in [0, 1]$.  No auxiliary variable needed.


In [ ]:
import jax
jax.config.update('jax_default_matmul_precision', 'float32')

import jax.numpy as jnp
import numpy as np
import optax
from jax import jvp, value_and_grad
from flax import linen as nn
from typing import Sequence
from functools import partial
from tqdm.auto import trange
import matplotlib.pyplot as plt


## Hyperparameters

In [ ]:
SEED       = 111
NC         = 128
NC_TEST    = 100
LR         = 1e-5
EPOCHS     = 5_000               # bump to 150_000 for paper-quality results
N_LAYERS   = 4
FEATURES   = 128
R          = 128
OUT_DIM    = 2                   # (u, θ)
X_MIN, X_MAX = 0.0, 11 * np.pi
T_MAX      = 1.0


## SPINN model + HVP

In [ ]:
# Forward-over-forward HVP.  Used to compute u_xx, u_tt, u_xxxx, etc.
def hvp_fwdfwd(f, primals, tangents, return_primals=False):
    g = lambda primals: jvp(f, (primals,), tangents)[1]
    primals_out, tangents_out = jvp(g, primals, tangents)
    if return_primals:
        return primals_out, tangents_out
    return tangents_out


In [ ]:
# Separable PINN with 2 axes and `out_dim` separate output fields.
# Each output i uses its own block of size r in the last linear layer; we
# split the rank-(r * out_dim) tensor and merge with einsum.
class SPINN2dMulti(nn.Module):
    features: Sequence[int]
    r: int
    out_dim: int
    mlp: str = 'modified_mlp'

    @nn.compact
    def __call__(self, t, x):
        inputs, outputs, preds = [t, x], [], []
        init = nn.initializers.glorot_normal()
        for X in inputs:
            if self.mlp == 'mlp':
                for fs in self.features[:-1]:
                    X = nn.tanh(nn.Dense(fs, kernel_init=init)(X))
                X = nn.Dense(self.r * self.out_dim, kernel_init=init)(X)
            else:
                U = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                V = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                H = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                for fs in self.features[:-1]:
                    Z = nn.tanh(nn.Dense(fs, kernel_init=init)(H))
                    H = (1 - Z) * U + Z * V
                X = nn.Dense(self.r * self.out_dim, kernel_init=init)(H)
            outputs.append(jnp.transpose(X, (1, 0)))  # (r*out_dim, N_axis)
        for i in range(self.out_dim):
            a = outputs[0][self.r * i:self.r * (i + 1)]  # (r, T)
            b = outputs[1][self.r * i:self.r * (i + 1)]  # (r, Nx)
            preds.append(jnp.einsum('rt,rx->tx', a, b))
        return preds if self.out_dim > 1 else preds[0]


## Analytic solution & data generator

In [ ]:
def exact_u(t, x):
    return 11 * jnp.pi * jnp.cos(t) * jnp.sin(x) * 0.5

def exact_theta(t, x):
    return (11 * 0.5 * jnp.pi * jnp.cos(x) + (x - 11 * jnp.pi / 2)) * jnp.cos(t)

def source_term(t, x):
    return jnp.cos(t)

def make_train_data(nc, key):
    keys = jax.random.split(key, 2)
    tc = jnp.linspace(0, T_MAX, nc + 2)[1:-1].reshape(-1, 1)
    xc = jnp.linspace(X_MIN, X_MAX, nc + 2)[1:-1].reshape(-1, 1)
    tm, xm = jnp.meshgrid(tc.ravel(), xc.ravel(), indexing='ij')
    uc = jnp.broadcast_to(source_term(tm, xm), tm.shape)

    ti = jnp.zeros((nc, 1))
    xi = jax.random.uniform(keys[1], (nc, 1), minval=X_MIN, maxval=X_MAX)
    ti_m, xi_m = jnp.meshgrid(ti.ravel(), xi.ravel(), indexing='ij')
    ui = exact_u(ti_m, xi_m); thetai = exact_theta(ti_m, xi_m)

    tb  = jax.random.uniform(keys[0], (nc, 1), minval=0., maxval=T_MAX)
    xbl = jnp.zeros((nc, 1)); xbr = X_MAX * jnp.ones((nc, 1))
    tbl_m, xbl_m = jnp.meshgrid(tb.ravel(), xbl.ravel(), indexing='ij')
    tbr_m, xbr_m = jnp.meshgrid(tb.ravel(), xbr.ravel(), indexing='ij')
    ubl = exact_u(tbl_m, xbl_m); ubr = exact_u(tbr_m, xbr_m)
    thetabl = exact_theta(tbl_m, xbl_m); thetabr = exact_theta(tbr_m, xbr_m)

    W = jnp.tril(jnp.ones((nc, nc)), k=-1)
    return (tc, xc, uc, ti, xi, ui, thetai,
            tb, xbl, xbr, ubl, ubr, thetabl, thetabr, W)


## Causal loss (coupled system)

In [ ]:
@partial(jax.jit, static_argnames=('apply_fn',))
def loss_and_grad(apply_fn, params, *train_data):
    (tc, xc, uc, ti, xi, ui, thetai,
     tb, xbl, xbr, ubl, ubr, thetabl, thetabr, W) = train_data

    def residual_loss(p):
        u, theta = apply_fn(p, tc, xc)
        v = jnp.ones(tc.shape)
        ux       = jvp(lambda x: apply_fn(p, tc, x)[0], (xc,), (v,))[1]
        thetax   = jvp(lambda x: apply_fn(p, tc, x)[1], (xc,), (v,))[1]
        utt      = hvp_fwdfwd(lambda t: apply_fn(p, t, xc)[0], (tc,), (v,))
        uxx      = hvp_fwdfwd(lambda x: apply_fn(p, tc, x)[0], (xc,), (v,))
        thetatt  = hvp_fwdfwd(lambda t: apply_fn(p, t, xc)[1], (tc,), (v,))
        thetaxx  = hvp_fwdfwd(lambda x: apply_fn(p, tc, x)[1], (xc,), (v,))

        res1 = (thetatt - thetaxx - (ux - theta))**2
        res2 = (utt - (uxx - thetax) - uc + u)**2
        loss_time = jnp.mean(res1 + res2, axis=1, keepdims=True)
        agg = jax.lax.stop_gradient(jnp.dot(W, loss_time))
        causal_w = jnp.exp(-3.0 * agg)
        return jnp.mean(causal_w * loss_time)

    def initial_loss(p):
        u0, th0 = apply_fn(p, ti, xi)
        l1 = jnp.mean((u0 - ui)**2) + jnp.mean((th0 - thetai)**2)
        v_t = jnp.ones(ti.shape)
        u_t0  = jvp(lambda t: apply_fn(p, t, xi)[0], (ti,), (v_t,))[1]
        th_t0 = jvp(lambda t: apply_fn(p, t, xi)[1], (ti,), (v_t,))[1]
        return l1 + jnp.mean(u_t0**2) + jnp.mean(th_t0**2)

    def boundary_loss(p):
        ul, thl = apply_fn(p, tb, xbl); ur, thr = apply_fn(p, tb, xbr)
        return (jnp.mean((ul - ubl)**2) + jnp.mean((ur - ubr)**2)
              + jnp.mean((thl - thetabl)**2) + jnp.mean((thr - thetabr)**2))

    total = lambda p: residual_loss(p) + initial_loss(p) + boundary_loss(p)
    return value_and_grad(total)(params)


## Initialize and train

In [ ]:
key = jax.random.PRNGKey(SEED)
key, sub_init, sub_data = jax.random.split(key, 3)

model = SPINN2dMulti(features=[FEATURES] * N_LAYERS, r=R, out_dim=OUT_DIM)
t0 = jnp.ones((NC, 1)); x0 = jnp.ones((NC, 1))
params = model.init(sub_init, t0, x0)
apply_fn = jax.jit(model.apply)

n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total trainable params: {n_params}")

train_data = make_train_data(NC, sub_data)

# Eval grid + ground truth (for periodic best-tracking on u)
t_eval = jnp.linspace(0, T_MAX, 100).reshape(-1, 1)
x_eval = jnp.linspace(X_MIN, X_MAX, 100).reshape(-1, 1)
tm_eval, xm_eval = jnp.meshgrid(t_eval.ravel(), x_eval.ravel(), indexing='ij')
u_true_eval  = exact_u(tm_eval, xm_eval)
th_true_eval = exact_theta(tm_eval, xm_eval)

optim = optax.adam(LR); state = optim.init(params)

losses = []
best_loss = 1e9
best_err = 1e9      # tracks u error
best_err_th = 1e9
best_params = params

pbar = trange(EPOCHS)
for e in pbar:
    loss, grads = loss_and_grad(apply_fn, params, *train_data)
    updates, state = optim.update(grads, state, params)
    params = optax.apply_updates(params, updates)
    losses.append(float(loss))
    if float(loss) <= best_loss:                       
        best_loss = float(loss)
        u_pred_eval, th_pred_eval = apply_fn(params, t_eval, x_eval)
        best_err    = float(jnp.linalg.norm(u_pred_eval  - u_true_eval)  / jnp.linalg.norm(u_true_eval))
        best_err_th = float(jnp.linalg.norm(th_pred_eval - th_true_eval) / jnp.linalg.norm(th_true_eval))
        best_params = params
    if (e + 1) % 500 == 0:
        pbar.set_postfix(loss=f"{loss:.3e}", err_u_at_best=f"{best_err:.3e}")

print(f"\nRel L2 (u) at best-loss step:     {best_err:.3e}")
print(f"Rel L2 (theta) at best-loss step: {best_err_th:.3e}")


## Evaluate and plot (using BEST params)

In [ ]:
# Relative L2 error against the analytic solution.
def relative_l2(pred, true):
    return float(jnp.linalg.norm(pred - true) / jnp.linalg.norm(true))

t_test = jnp.linspace(0, T_MAX, 100).reshape(-1, 1)
x_test = jnp.linspace(X_MIN, X_MAX, 100).reshape(-1, 1)
tm, xm = jnp.meshgrid(t_test.ravel(), x_test.ravel(), indexing='ij')
u_true = exact_u(tm, xm); th_true = exact_theta(tm, xm)
u_pred, th_pred = apply_fn(best_params, t_test, x_test)

err_u  = best_err
err_th = best_err_th
print(f"Best relative L2 error (u):     {err_u:.3e}")
print(f"Best relative L2 error (theta): {err_th:.3e}")

fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for row, (true, pred, name) in enumerate([
        (u_true,  u_pred,  'u'),
        (th_true, th_pred, r'$\theta$')]):
    for col, (data, title) in enumerate(zip(
            [true, pred, jnp.abs(true - pred)],
            [f'Exact {name}', f'Predicted {name}', f'|Error| {name}'])):
        im = axs[row, col].pcolormesh(np.asarray(tm), np.asarray(xm), np.asarray(data),
                                     cmap='RdBu_r', shading='auto')
        axs[row, col].set_xlabel('t'); axs[row, col].set_ylabel('x')
        axs[row, col].set_title(title); plt.colorbar(im, ax=axs[row, col])
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.semilogy(losses); plt.xlabel('epoch'); plt.ylabel('total loss')
plt.title('Timoshenko Causal SPINN — training loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
